In [0]:
import logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.window import Window
from delta.tables import DeltaTable
 
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)
 
spark = SparkSession.builder.getOrCreate()
 
MODEL_NAME = "gbt_price_predictor_v1"

# 1 — Read Gold Fact + Dims
log.info("Reading Gold layer tables...")
 
fact = spark.table("crypto_catalog.gold.fact_crypto_daily")
dim_coin = spark.table("crypto_catalog.gold.dim_coin")
dim_date = spark.table("crypto_catalog.gold.dim_date")
 
log.info(f"fact_crypto_daily rows: {fact.count():,}")

In [0]:
# 2 — Build Training Dataset 

log.info("Building feature set...")
 
w = Window.partitionBy("coin_key").orderBy("trade_date")
 
OPTIONAL_COLS = [
    "avg_sentiment_score", "post_count", "avg_engagement_score",
    "active_addresses", "transaction_count",
    "avg_network_fees", "net_exchange_flow", "bullish_score"
]
 
df_ml = (
    fact
    .withColumn("target_next_close", F.lead("close_price", 1).over(w))
    .filter(F.col("target_next_close").isNotNull())
    .filter(
        F.col("ma_7").isNotNull() &
        F.col("ma_14").isNotNull() &
        F.col("ma_30").isNotNull() &
        F.col("volatility_7d").isNotNull() &
        F.col("price_change_pct").isNotNull()
    )
    .withColumn("has_live_signal", F.col("avg_sentiment_score").isNotNull().cast("int"))
    .fillna(0, subset=OPTIONAL_COLS)
    # Scale-free target: % change to next day, not raw price
    .withColumn(
        "target_pct_change",
        (F.col("target_next_close") - F.col("close_price")) / F.col("close_price")
    )
)
 
df_ml = (
    df_ml
    .withColumn("ma_7_ratio",  F.col("ma_7")  / F.col("close_price"))
    .withColumn("ma_14_ratio", F.col("ma_14") / F.col("close_price"))
    .withColumn("ma_30_ratio", F.col("ma_30") / F.col("close_price"))
    .withColumn("volatility_ratio", F.col("volatility_7d") / F.col("close_price"))
    .withColumn("price_range_ratio", F.col("price_range") / F.col("close_price"))
)
 
FEATURE_COLS = [
    "ma_7_ratio", "ma_14_ratio", "ma_30_ratio",
    "price_change_pct", "price_range_ratio", "volatility_ratio",
    "avg_sentiment_score", "post_count", "avg_engagement_score",
    "active_addresses", "transaction_count",
    "avg_network_fees", "net_exchange_flow",
    "bullish_score", "has_live_signal"
]
 
log.info(f"Training rows: {df_ml.count():,}")
log.info(f"Features used: {len(FEATURE_COLS)}")
 

In [0]:
# 3 — Train / Test Split + Train Model
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
log.info(f"Train: {train_df.count():,} | Test: {test_df.count():,}")
 
assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="features",
    handleInvalid="skip"
)
 
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="target_pct_change",
    maxIter=50,
    maxDepth=5,
    seed=42
)
 
log.info("Training GBT Regressor...")
train_assembled = assembler.transform(train_df)
model = gbt.fit(train_assembled)
log.info("Training complete")

In [0]:
# 4 — Evaluate
log.info("Evaluating model on test set...")
test_assembled = assembler.transform(test_df)
predictions = model.transform(test_assembled)
 
evaluator_rmse = RegressionEvaluator(
    labelCol="target_pct_change", predictionCol="prediction", metricName="rmse"
)
evaluator_mae = RegressionEvaluator(
    labelCol="target_pct_change", predictionCol="prediction", metricName="mae"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="target_pct_change", predictionCol="prediction", metricName="r2"
)
 
rmse = evaluator_rmse.evaluate(predictions)
mae  = evaluator_mae.evaluate(predictions)
r2   = evaluator_r2.evaluate(predictions)
 
log.info("=" * 55)
log.info("MODEL EVALUATION")
log.info("=" * 55)
log.info(f"RMSE : {rmse:.4f}")
log.info(f"MAE  : {mae:.4f}")
log.info(f"R2   : {r2:.4f}")
 
display(
    predictions
    .withColumn("predicted_next_close", F.col("close_price") * (1 + F.col("prediction")))
    .select(
        "symbol", "trade_date", "close_price",
        "target_next_close", "prediction", "predicted_next_close"
    ).orderBy("symbol", "trade_date").limit(20)
)

In [0]:
# 5 — Generate Predictions for Full Dataset
log.info("Generating predictions for full dataset...")
 
full_assembled = assembler.transform(df_ml)
full_predictions = model.transform(full_assembled)
 
predictions_final = (
    full_predictions
    .withColumn("predicted_next_close", F.col("close_price") * (1 + F.col("prediction")))
    .select(
        "coin_key",
        "date_key",
        "symbol",
        "trade_date",
        F.col("close_price").alias("actual_close_price"),
        F.col("target_next_close").alias("actual_next_close"),
        "predicted_next_close",
        F.lit(MODEL_NAME).alias("model_name"),
        F.current_timestamp().alias("prediction_made_at")
    )
)
 
log.info(f"Predictions generated: {predictions_final.count():,} rows")

In [0]:
# 6 — Save fact_price_predictions (MERGE)
spark.sql("CREATE SCHEMA IF NOT EXISTS crypto_catalog.gold")
 
if spark.catalog.tableExists("crypto_catalog.gold.fact_price_predictions"):
    log.info("Predictions table exists — running MERGE...")
    (
        DeltaTable.forName(spark, "crypto_catalog.gold.fact_price_predictions")
        .alias("target")
        .merge(
            predictions_final.alias("source"),
            "target.symbol = source.symbol AND target.trade_date = source.trade_date AND target.model_name = source.model_name"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    log.info("MERGE complete")
else:
    log.info("Creating fact_price_predictions (first run)...")
    predictions_final.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("crypto_catalog.gold.fact_price_predictions")
    log.info("fact_price_predictions created")
 

In [0]:
# CELL 7 — Verify
log.info("=" * 55)
log.info("ML LAYER COMPLETE")
log.info("=" * 55)
log.info(f"fact_price_predictions rows: {spark.table('crypto_catalog.gold.fact_price_predictions').count():,}")
log.info(f"Model: {MODEL_NAME} | RMSE: {rmse:.4f} | R2: {r2:.4f}")
 
display(spark.sql("SHOW TABLES IN crypto_catalog.gold"))

In [0]:
df_check = spark.table("crypto_catalog.gold.fact_price_predictions")

display(
    df_check.select(
        "symbol", "trade_date",
        "actual_close_price", "actual_next_close", "predicted_next_close",
        "model_name"
    ).orderBy("symbol", "trade_date")
)

In [0]:
df_eval = (
    df_check
    .withColumn("abs_pct_error",
        F.abs(F.col("predicted_next_close") - F.col("actual_next_close")) / F.col("actual_next_close") * 100
    )
)

df_eval.select(
    F.avg("abs_pct_error").alias("mean_abs_pct_error"),
    F.expr("percentile_approx(abs_pct_error, 0.5)").alias("median_abs_pct_error")
).show()

In [0]:
display(
    df_eval.orderBy(F.desc("abs_pct_error"))
    .select("symbol", "trade_date", "actual_close_price", "actual_next_close", "predicted_next_close", "abs_pct_error")
    .limit(20)
)


display(
    df_eval.groupBy("symbol")
    .agg(
        F.avg("abs_pct_error").alias("mean_err"),
        F.expr("percentile_approx(abs_pct_error, 0.5)").alias("median_err"),
        F.min("actual_close_price").alias("min_price")
    )
    .orderBy(F.desc("mean_err"))
)